# Historical Buying Opportunities Analysis

Looking back at the last 5 years to identify:
1. What were the best buying opportunities (in hindsight)?
2. What were SOPR/STH-SOPR values at those times?
3. What patterns can we learn from?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Historical Buying Opportunities Analysis 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(mvrv, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

# Last 5 years
df = df[df.index >= '2020-01-01'].copy()
df = df.dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

---
## 1. Identify Major Price Bottoms (Hindsight)

In [ ]:
# Find local minima - points that were lower than surrounding 30 days
df['rolling_min_30'] = df['price'].rolling(61, center=True).min()
df['is_local_min'] = df['price'] == df['rolling_min_30']

# Also calculate forward returns to see which bottoms were actually good buys
df['fwd_30d'] = df['price'].shift(-30) / df['price'] - 1
df['fwd_90d'] = df['price'].shift(-90) / df['price'] - 1
df['fwd_180d'] = df['price'].shift(-180) / df['price'] - 1
df['fwd_365d'] = df['price'].shift(-365) / df['price'] - 1

# Distance from ATH
df['ath'] = df['price'].cummax()
df['drawdown'] = (df['price'] - df['ath']) / df['ath']

print("Calculated forward returns and drawdowns")

In [ ]:
# Known major buying opportunities (hindsight)
major_bottoms = [
    ('2020-03-12', 'COVID Crash', 'Black Thursday - massive liquidations'),
    ('2020-03-13', 'COVID Crash +1', 'Continuation of crash'),
    ('2021-05-19', 'China Ban Crash', 'Mining ban + Elon FUD'),
    ('2021-06-22', 'Summer Bottom', 'Continuation of May crash'),
    ('2021-07-20', 'July Bottom', 'Final capitulation before run to ATH'),
    ('2021-12-04', 'Dec Flash Crash', 'Leverage flush'),
    ('2022-01-24', 'Jan 2022 Low', 'Start of bear market'),
    ('2022-05-12', 'LUNA Crash', 'UST depeg catastrophe'),
    ('2022-06-18', 'June 2022 Low', 'Celsius/3AC contagion'),
    ('2022-11-09', 'FTX Crash', 'FTX collapse'),
    ('2022-11-21', 'Cycle Bottom', 'Actual cycle low ~$15.5k'),
    ('2023-03-10', 'SVB Crash', 'Banking crisis dip'),
    ('2023-06-15', 'Summer 2023 Dip', 'SEC lawsuit dip'),
    ('2023-09-11', 'Sept 2023 Low', 'Pre-ETF accumulation'),
    ('2024-01-23', 'ETF Sell News', 'Sell the news after ETF approval'),
    ('2024-05-01', 'May 2024 Dip', 'Post-halving correction'),
    ('2024-07-05', 'July 2024 Dip', 'Mt Gox fears'),
    ('2024-08-05', 'Yen Carry Crash', 'Global macro crash'),
    ('2024-09-06', 'Sept 2024 Low', 'Pre-election dip'),
]

print(f"Analyzing {len(major_bottoms)} known buying opportunities")

In [ ]:
# Analyze each bottom
print("MAJOR BUYING OPPORTUNITIES - SOPR VALUES")
print("="*140)
print(f"{'Date':<12} {'Event':<20} {'Price':>10} {'SOPR':>8} {'STH-SOPR':>10} {'MVRV':>8} {'DD%':>8} {'30d Fwd':>10} {'90d Fwd':>10} {'365d Fwd':>10}")
print("-"*140)

bottom_data = []

for date_str, name, desc in major_bottoms:
    try:
        # Find closest date in data
        target = pd.Timestamp(date_str, tz='UTC')
        idx = df.index.get_indexer([target], method='nearest')[0]
        row = df.iloc[idx]
        actual_date = df.index[idx]
        
        bottom_data.append({
            'date': actual_date,
            'name': name,
            'price': row['price'],
            'sopr': row['sopr'],
            'sth_sopr': row['sopr_sth'],
            'mvrv': row['mvrv'],
            'drawdown': row['drawdown'],
            'fwd_30d': row['fwd_30d'],
            'fwd_90d': row['fwd_90d'],
            'fwd_365d': row['fwd_365d'],
        })
        
        fwd30 = f"{row['fwd_30d']*100:+.0f}%" if pd.notna(row['fwd_30d']) else 'N/A'
        fwd90 = f"{row['fwd_90d']*100:+.0f}%" if pd.notna(row['fwd_90d']) else 'N/A'
        fwd365 = f"{row['fwd_365d']*100:+.0f}%" if pd.notna(row['fwd_365d']) else 'N/A'
        
        print(f"{actual_date.strftime('%Y-%m-%d'):<12} {name:<20} ${row['price']:>9,.0f} {row['sopr']:>8.3f} {row['sopr_sth']:>10.3f} {row['mvrv']:>8.2f} {row['drawdown']*100:>7.0f}% {fwd30:>10} {fwd90:>10} {fwd365:>10}")
    except Exception as e:
        print(f"{date_str:<12} {name:<20} Error: {e}")

bottoms_df = pd.DataFrame(bottom_data)

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("SOPR VALUES AT MAJOR BOTTOMS - SUMMARY")
print("="*70)

print(f"\nSOPR at bottoms:")
print(f"  Min:    {bottoms_df['sopr'].min():.3f}")
print(f"  Max:    {bottoms_df['sopr'].max():.3f}")
print(f"  Mean:   {bottoms_df['sopr'].mean():.3f}")
print(f"  Median: {bottoms_df['sopr'].median():.3f}")

print(f"\nSTH-SOPR at bottoms:")
print(f"  Min:    {bottoms_df['sth_sopr'].min():.3f}")
print(f"  Max:    {bottoms_df['sth_sopr'].max():.3f}")
print(f"  Mean:   {bottoms_df['sth_sopr'].mean():.3f}")
print(f"  Median: {bottoms_df['sth_sopr'].median():.3f}")

print(f"\nMVRV at bottoms:")
print(f"  Min:    {bottoms_df['mvrv'].min():.2f}")
print(f"  Max:    {bottoms_df['mvrv'].max():.2f}")
print(f"  Mean:   {bottoms_df['mvrv'].mean():.2f}")

print(f"\n% of bottoms where SOPR < 1:     {(bottoms_df['sopr'] < 1).mean()*100:.0f}%")
print(f"% of bottoms where STH-SOPR < 1: {(bottoms_df['sth_sopr'] < 1).mean()*100:.0f}%")
print(f"% of bottoms where BOTH < 1:     {((bottoms_df['sopr'] < 1) & (bottoms_df['sth_sopr'] < 1)).mean()*100:.0f}%")

---
## 2. Visualize Price with SOPR

In [ ]:
# Create comprehensive chart
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.4, 0.2, 0.2, 0.2],
    subplot_titles=('BTC Price (Log)', 'SOPR', 'STH-SOPR', 'MVRV')
)

# Price
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='orange')), row=1, col=1)

# Mark bottoms on price
for _, b in bottoms_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[b['date']], y=[b['price']], 
        mode='markers', marker=dict(size=12, color='green', symbol='triangle-up'),
        name=b['name'], showlegend=False,
        hovertext=f"{b['name']}<br>SOPR: {b['sopr']:.3f}<br>STH: {b['sth_sopr']:.3f}"
    ), row=1, col=1)

# SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr'], name='SOPR', line=dict(color='blue')), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=2, col=1)

# STH-SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr_sth'], name='STH-SOPR', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=3, col=1)

# MVRV
fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV', line=dict(color='green')), row=4, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=4, col=1)
fig.add_hline(y=2, line_dash='dash', line_color='orange', row=4, col=1)
fig.add_hline(y=3, line_dash='dash', line_color='red', row=4, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=1000, title='BTC Price vs On-Chain Metrics (Last 5 Years)', showlegend=False)
fig.show()

---
## 3. SOPR Distribution Analysis

In [ ]:
# What % of time is SOPR below certain levels?
print("SOPR DISTRIBUTION (Last 5 Years)")
print("="*60)

thresholds = [0.90, 0.92, 0.94, 0.96, 0.98, 1.00, 1.02]

print(f"\n{'Threshold':<15} {'% Days Below':>15} {'Occurrences':>15}")
print("-"*50)
for t in thresholds:
    pct = (df['sopr'] < t).mean() * 100
    count = (df['sopr'] < t).sum()
    print(f"SOPR < {t:<8} {pct:>14.1f}% {count:>15}")

print(f"\n{'Threshold':<15} {'% Days Below':>15} {'Occurrences':>15}")
print("-"*50)
for t in thresholds:
    pct = (df['sopr_sth'] < t).mean() * 100
    count = (df['sopr_sth'] < t).sum()
    print(f"STH-SOPR < {t:<5} {pct:>14.1f}% {count:>15}")

In [ ]:
# Forward returns when SOPR is below threshold
print("\nFORWARD RETURNS BY SOPR LEVEL")
print("="*80)

print(f"\n{'Condition':<25} {'Avg 30d':>12} {'Avg 90d':>12} {'Avg 365d':>12} {'Count':>10}")
print("-"*80)

conditions = [
    ('All days', df['sopr'] > 0),
    ('SOPR < 1', df['sopr'] < 1),
    ('SOPR < 0.98', df['sopr'] < 0.98),
    ('SOPR < 0.96', df['sopr'] < 0.96),
    ('SOPR < 0.95', df['sopr'] < 0.95),
    ('STH-SOPR < 1', df['sopr_sth'] < 1),
    ('STH-SOPR < 0.98', df['sopr_sth'] < 0.98),
    ('STH-SOPR < 0.96', df['sopr_sth'] < 0.96),
    ('STH-SOPR < 0.95', df['sopr_sth'] < 0.95),
    ('SOPR<1 & STH<1', (df['sopr'] < 1) & (df['sopr_sth'] < 1)),
    ('SOPR<0.98 & STH<0.98', (df['sopr'] < 0.98) & (df['sopr_sth'] < 0.98)),
]

for name, cond in conditions:
    subset = df[cond]
    if len(subset) > 0:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        avg_365 = subset['fwd_365d'].mean() * 100
        print(f"{name:<25} {avg_30:>+11.1f}% {avg_90:>+11.1f}% {avg_365:>+11.1f}% {len(subset):>10}")

---
## 4. Best Bottoms Deep Dive

In [ ]:
# Rank bottoms by forward returns
print("BOTTOMS RANKED BY 365-DAY FORWARD RETURN")
print("="*100)

valid_bottoms = bottoms_df[bottoms_df['fwd_365d'].notna()].copy()
valid_bottoms = valid_bottoms.sort_values('fwd_365d', ascending=False)

print(f"\n{'Rank':<6} {'Date':<12} {'Event':<20} {'365d Return':>12} {'SOPR':>8} {'STH-SOPR':>10}")
print("-"*80)

for i, (_, row) in enumerate(valid_bottoms.iterrows()):
    sopr_flag = '✅' if row['sopr'] < 1 else '❌'
    sth_flag = '✅' if row['sth_sopr'] < 1 else '❌'
    print(f"{i+1:<6} {row['date'].strftime('%Y-%m-%d'):<12} {row['name']:<20} {row['fwd_365d']*100:>+11.0f}% {row['sopr']:>7.3f} {sopr_flag} {row['sth_sopr']:>7.3f} {sth_flag}")

In [ ]:
# Check signal accuracy
print("\nSIGNAL ACCURACY AT MAJOR BOTTOMS")
print("="*70)

sopr_correct = (bottoms_df['sopr'] < 1).sum()
sth_correct = (bottoms_df['sth_sopr'] < 1).sum()
both_correct = ((bottoms_df['sopr'] < 1) & (bottoms_df['sth_sopr'] < 1)).sum()
total = len(bottoms_df)

print(f"\nSOPR < 1 at bottoms:     {sopr_correct}/{total} ({sopr_correct/total*100:.0f}%)")
print(f"STH-SOPR < 1 at bottoms: {sth_correct}/{total} ({sth_correct/total*100:.0f}%)")
print(f"Both < 1 at bottoms:     {both_correct}/{total} ({both_correct/total*100:.0f}%)")

# Which bottoms did we MISS?
print("\n🚨 BOTTOMS WHERE SOPR >= 1 (signal missed):")
missed = bottoms_df[bottoms_df['sopr'] >= 1]
for _, row in missed.iterrows():
    print(f"   {row['date'].strftime('%Y-%m-%d')} {row['name']}: SOPR={row['sopr']:.3f}, STH={row['sth_sopr']:.3f}")

print("\n🚨 BOTTOMS WHERE STH-SOPR >= 1 (signal missed):")
missed = bottoms_df[bottoms_df['sth_sopr'] >= 1]
for _, row in missed.iterrows():
    print(f"   {row['date'].strftime('%Y-%m-%d')} {row['name']}: SOPR={row['sopr']:.3f}, STH={row['sth_sopr']:.3f}")

---
## 5. Find ALL Days with SOPR < 1

In [ ]:
# All periods where both SOPR and STH-SOPR < 1
df['both_below_1'] = (df['sopr'] < 1) & (df['sopr_sth'] < 1)

# Find contiguous periods
df['period_start'] = df['both_below_1'] & ~df['both_below_1'].shift(1).fillna(False)
df['period_id'] = df['period_start'].cumsum()
df.loc[~df['both_below_1'], 'period_id'] = 0

periods = []
for pid in df[df['period_id'] > 0]['period_id'].unique():
    period_data = df[df['period_id'] == pid]
    periods.append({
        'start': period_data.index[0],
        'end': period_data.index[-1],
        'days': len(period_data),
        'price_start': period_data['price'].iloc[0],
        'price_min': period_data['price'].min(),
        'sopr_min': period_data['sopr'].min(),
        'sth_min': period_data['sopr_sth'].min(),
    })

periods_df = pd.DataFrame(periods)

print(f"PERIODS WHERE BOTH SOPR AND STH-SOPR < 1")
print("="*120)
print(f"\n{'Start':<12} {'End':<12} {'Days':>6} {'Price Range':>20} {'Min SOPR':>10} {'Min STH':>10}")
print("-"*80)

for _, p in periods_df.iterrows():
    print(f"{p['start'].strftime('%Y-%m-%d'):<12} {p['end'].strftime('%Y-%m-%d'):<12} {p['days']:>6} ${p['price_min']:>8,.0f}-{p['price_start']:>8,.0f} {p['sopr_min']:>10.3f} {p['sth_min']:>10.3f}")

print(f"\nTotal periods: {len(periods_df)}")
print(f"Total days with signal: {df['both_below_1'].sum()}")

---
## 6. Optimal SOPR Threshold Analysis

In [ ]:
# Test different SOPR thresholds
print("OPTIMAL SOPR THRESHOLD ANALYSIS")
print("="*100)
print(f"\n{'SOPR Threshold':<18} {'Days':>8} {'Avg 30d':>12} {'Avg 90d':>12} {'Avg 365d':>12} {'Signal Quality':>15}")
print("-"*100)

for t in [0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 1.00, 1.01, 1.02]:
    cond = df['sopr'] < t
    subset = df[cond]
    if len(subset) > 10:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        avg_365 = subset['fwd_365d'].mean() * 100
        
        # Signal quality = avg return / frequency (higher is better)
        quality = avg_90 / (len(subset) / len(df) * 100) if len(subset) > 0 else 0
        
        print(f"SOPR < {t:<11} {len(subset):>8} {avg_30:>+11.1f}% {avg_90:>+11.1f}% {avg_365:>+11.1f}% {quality:>15.2f}")

In [ ]:
# Same for STH-SOPR
print("\nOPTIMAL STH-SOPR THRESHOLD ANALYSIS")
print("="*100)
print(f"\n{'STH-SOPR Threshold':<18} {'Days':>8} {'Avg 30d':>12} {'Avg 90d':>12} {'Avg 365d':>12} {'Signal Quality':>15}")
print("-"*100)

for t in [0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 1.00, 1.01, 1.02]:
    cond = df['sopr_sth'] < t
    subset = df[cond]
    if len(subset) > 10:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        avg_365 = subset['fwd_365d'].mean() * 100
        quality = avg_90 / (len(subset) / len(df) * 100) if len(subset) > 0 else 0
        
        print(f"STH-SOPR < {t:<8} {len(subset):>8} {avg_30:>+11.1f}% {avg_90:>+11.1f}% {avg_365:>+11.1f}% {quality:>15.2f}")

---
## 7. Summary

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

print(f"""
📊 AT MAJOR BOTTOMS:
   • SOPR < 1 occurred at {(bottoms_df['sopr'] < 1).mean()*100:.0f}% of bottoms
   • STH-SOPR < 1 occurred at {(bottoms_df['sth_sopr'] < 1).mean()*100:.0f}% of bottoms
   • Average SOPR at bottoms: {bottoms_df['sopr'].mean():.3f}
   • Average STH-SOPR at bottoms: {bottoms_df['sth_sopr'].mean():.3f}

📈 FORWARD RETURNS (when signal is active):
   • SOPR < 1: avg 90d return = {df[df['sopr'] < 1]['fwd_90d'].mean()*100:+.1f}%
   • STH-SOPR < 1: avg 90d return = {df[df['sopr_sth'] < 1]['fwd_90d'].mean()*100:+.1f}%
   • Both < 1: avg 90d return = {df[(df['sopr'] < 1) & (df['sopr_sth'] < 1)]['fwd_90d'].mean()*100:+.1f}%

🔍 SIGNAL FREQUENCY:
   • SOPR < 1: {(df['sopr'] < 1).mean()*100:.1f}% of days
   • STH-SOPR < 1: {(df['sopr_sth'] < 1).mean()*100:.1f}% of days
   • Both < 1: {((df['sopr'] < 1) & (df['sopr_sth'] < 1)).mean()*100:.1f}% of days
""")